# Synthetic Data Generation

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

In [6]:
# Check exactly where Jupyter sees cwd
current_dir = os.getcwd()
print(f"1. Jupyter is currently running in: {current_dir}")

# Only go up if actually inside the notebooks folder
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    # If already in the root, stay
    project_root = current_dir

print(f"2. Project Root identified as: {project_root}")

# Build the final path
raw_data_dir = os.path.join(project_root, 'data', 'raw')
print(f"3. Creating data directory at: {raw_data_dir}")

# Safely create the directory
os.makedirs(raw_data_dir, exist_ok=True)

# Configuration for a 13,000+ row dataset
start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 4, 30)
dates = pd.date_range(start_date, end_date)

# Constants for synthetic integrity
account_codes = [f"{np.random.randint(10000, 99999)}" for _ in range(15)]
vehicle_types = ['SEDN', 'SUV', 'Sprinter', 'MINI', 'MID-MINI', 'MC']
states = ['IL', 'NY', 'MA', 'CA', 'DC', 'TX', 'GA', 'FL', 'SC', 'NJ', 'PA', 'OH']
v_weights = [0.45, 0.30, 0.12, 0.08, 0.03, 0.02]

# Financial mapping (Min Rev, Max Rev, Base Service Cost, Base Overhead)
fin_logic = {
    'SEDN': (120, 180, 40, 15),
    'SUV': (180, 320, 60, 25),
    'Sprinter': (450, 750, 150, 75),
    'MINI': (700, 1100, 250, 100),
    'MID-MINI': (900, 1500, 350, 150),
    'MC': (1600, 2500, 800, 200)
}

hour_probs = np.array([0.05, 0.02, 0.01, 0.01, 0.03, 0.08, 0.12, 0.15, 0.10, 0.05, 0.04, 0.04, 0.04, 0.04, 0.04, 0.05, 0.05, 0.05, 0.03, 0.02, 0.02, 0.02, 0.01, 0.02])
hour_probs /= hour_probs.sum()

data = []
res_no_counter = 100001

for current_date in dates:
    # Baseline volume with Gamma distribution
    base_trips = np.random.gamma(shape=3.0, scale=4.5) 
    
    # Weekly seasonality
    day_of_week = current_date.weekday()
    season_mult = {0: 1.3, 1: 1.4, 2: 1.3, 3: 1.1, 4: 0.9, 5: 0.7, 6: 0.5}[day_of_week]
    
    # Oct 2025 Trend Shift
    trend_mult = 1.3 if current_date >= datetime(2025, 10, 1) else 1.0
    event_chance = 0.15 if current_date >= datetime(2025, 10, 1) else 0.08
    
    num_trips = int(base_trips * season_mult * trend_mult)
    
    if np.random.random() < event_chance:
        num_trips = int(num_trips * np.random.uniform(1.6, 2.8))
        event_code = str(np.random.randint(20000, 89999))
    else:
        event_code = "Standard"

    for _ in range(num_trips):
        v_type = np.random.choice(vehicle_types, p=v_weights)
        s_type = np.random.choice(['In-House', 'Affiliate'], p=[0.6, 0.4])
        duration = np.random.randint(30, 180)
        
        # Timing
        hour = np.random.choice(range(24), p=hour_probs)
        pickup_time = current_date.replace(hour=hour, minute=np.random.randint(0, 60))
        dropoff_time = pickup_time + timedelta(minutes=duration)
        
        # Financials
        rev_min, rev_max, sc_base, oh_base = fin_logic[v_type]
        revenue = round(np.random.uniform(rev_min, rev_max), 2)
        overhead = round(oh_base + np.random.uniform(-5, 15), 2)
        service_cost = round(sc_base * np.random.uniform(0.8, 1.2), 2)
        
        int_cost, aff_cost = (service_cost, 0.0) if s_type == 'In-House' else (0.0, service_cost)
        total_cost = round(service_cost + overhead, 2)
        
        data.append([
            res_no_counter, np.random.choice(account_codes),
            pickup_time.strftime('%b %d %Y %I:%M%p'),
            dropoff_time.strftime('%b %d %Y %I:%M%p'),
            v_type, np.random.choice(states), event_code,
            revenue, int_cost, aff_cost, total_cost,
            round(revenue - total_cost, 2), s_type,
            round(duration / 60 + 0.5, 1), overhead,
            np.random.choice(['Actual', 'Estimate'], p=[0.95, 0.05])
        ])
        res_no_counter += 1

df = pd.DataFrame(data, columns=[
    'ResNo', 'AccountCode', 'PickupDateTimeScheduled', 'DropoffDateTimeActual',
    'PrefferedVehicleType', 'Pickup_State', 'EventCategory', 'Revenue',
    'InternalDriverCost', 'AffiliateCost', 'TotalCost', 'GrossProfit',
    'ServiceType', 'EstDriverHours', 'EstOverheadCost', 'RevenueStatus'
])

output_path = os.path.join(raw_data_dir, 'synthetic_mobility_data.csv')
df.to_csv(output_path, index=False)
print(f"Success.. {len(df)} rows generated securely in {output_path}")

1. Jupyter is currently running in: C:\Users\Eric\Desktop\Mobility-Demand-Forecaster
2. Project Root identified as: C:\Users\Eric\Desktop\Mobility-Demand-Forecaster
3. Creating data directory at: C:\Users\Eric\Desktop\Mobility-Demand-Forecaster\data\raw
Success.. 14705 rows generated securely in C:\Users\Eric\Desktop\Mobility-Demand-Forecaster\data\raw\synthetic_mobility_data.csv
